# Part 2 — Cross-Domain Acne Classification (Google Colab)

**Before running:** Runtime → Change runtime type → **A100 GPU**

This notebook covers all of Part 2 end-to-end:
1. Setup (clone repo, install deps, download ACNE04)
2. Patch extraction — crop positive/negative patches from ACNE04 bounding boxes
3. Train EfficientNet-B0 classifier on ACNE04 patches
4. Download & prepare DermNet dataset
5. Evaluate on DermNet test set (Accuracy, F1, AUROC)
6. Grad-CAM visualizations on DermNet predictions
7. Reflection

Run cells top to bottom.

---
## Section 1 — Setup

In [ ]:
# Clone repo
import os

REPO_DIR = "/content/AcneDetection"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/EvxLee/AcneDetection.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print("Repo already exists — pulled latest.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
!pip install -q roboflow python-dotenv timm grad-cam scikit-learn
print("Dependencies installed.")

In [ ]:
# Set Roboflow credentials — paste your API key here
import os

os.environ["ROBOFLOW_API_KEY"]   = "YOUR_API_KEY_HERE"   # ← paste your key
os.environ["ROBOFLOW_WORKSPACE"] = "evan-lee-rrndd"
os.environ["ROBOFLOW_PROJECT"]   = "acne04-detection-p8j0d"
os.environ["ROBOFLOW_VERSION"]   = "1"

with open(f"{REPO_DIR}/.env", "w") as f:
    for k in ["ROBOFLOW_API_KEY", "ROBOFLOW_WORKSPACE", "ROBOFLOW_PROJECT", "ROBOFLOW_VERSION"]:
        f.write(f"{k}={os.environ[k]}\n")
print("Credentials set.")

In [ ]:
# Download ACNE04 dataset
!python part1_detection/roboflow_loader.py --download

In [ ]:
# Verify GPU
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
else:
    print("No GPU — go to Runtime → Change runtime type → A100")

---
## Section 2 — Patch Extraction

Creates a binary classification dataset from ACNE04 bounding boxes:

- **Positive (acne)**: crop each bounding box region, resize to 224×224
- **Negative (no_acne)**: randomly crop same-sized regions with zero overlap with any GT box

Output (standard PyTorch ImageFolder format):
```
data/patches/
├── train/
│   ├── acne/
│   └── no_acne/
└── val/
    ├── acne/
    └── no_acne/
```

In [ ]:
import json
import random
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

%matplotlib inline

DATA_DIR   = Path("data/acne04")
PATCH_DIR  = Path("data/patches")
PATCH_SIZE = 224
MIN_BOX    = 20
NEG_SIZE   = 90
SEED       = 42

SPLITS = {"train": "train", "valid": "val"}

random.seed(SEED)

In [ ]:
def is_valid_patch(patch_img, min_brightness=40, min_color_spread=15):
    """Reject dark/greyscale patches (backgrounds, hair, watermarks)."""
    arr = np.array(patch_img, dtype=float)
    if arr.mean() < min_brightness:
        return False
    channel_means = arr.mean(axis=(0, 1))
    if channel_means.max() - channel_means.min() < min_color_spread:
        return False
    return True

def has_overlap(box, gt_boxes):
    for g in gt_boxes:
        if box[0] < g[2] and box[2] > g[0] and box[1] < g[3] and box[3] > g[1]:
            return True
    return False

def sample_negative(img, img_w, img_h, gt_boxes, size, max_tries=150):
    for _ in range(max_tries):
        x1 = random.randint(0, max(0, img_w - size))
        y1 = random.randint(0, max(0, img_h - size))
        candidate = [x1, y1, x1 + size, y1 + size]
        if has_overlap(candidate, gt_boxes):
            continue
        patch = img.crop(candidate).resize((PATCH_SIZE, PATCH_SIZE), Image.BILINEAR)
        if is_valid_patch(patch):
            return candidate, patch
    return None, None

def extract_patches(acne04_split, patch_split):
    pos_dir = PATCH_DIR / patch_split / "acne"
    neg_dir = PATCH_DIR / patch_split / "no_acne"
    pos_dir.mkdir(parents=True, exist_ok=True)
    neg_dir.mkdir(parents=True, exist_ok=True)

    with open(DATA_DIR / acne04_split / "_annotations.coco.json") as f:
        coco = json.load(f)

    ann_map = {}
    for ann in coco["annotations"]:
        ann_map.setdefault(ann["image_id"], []).append(ann)

    pos_count = neg_count = skipped = rejected = 0

    for meta in coco["images"]:
        img_id = meta["id"]
        anns   = ann_map.get(img_id, [])
        if not anns:
            continue

        img      = Image.open(DATA_DIR / acne04_split / meta["file_name"]).convert("RGB")
        img_w, img_h = img.size
        gt_boxes = []

        for ann in anns:
            x, y, w, h = ann["bbox"]
            if w < MIN_BOX or h < MIN_BOX:
                skipped += 1
                continue
            x1, y1 = int(x), int(y)
            x2, y2 = min(int(x + w), img_w), min(int(y + h), img_h)
            gt_boxes.append([x1, y1, x2, y2])
            patch = img.crop((x1, y1, x2, y2)).resize((PATCH_SIZE, PATCH_SIZE), Image.BILINEAR)
            ann_id = ann["id"]
            patch.save(pos_dir / f"{img_id}_{ann_id}.jpg", quality=90)
            pos_count += 1

        for i in range(len(gt_boxes)):
            box, patch = sample_negative(img, img_w, img_h, gt_boxes, NEG_SIZE)
            if patch is None:
                rejected += 1
                continue
            patch.save(neg_dir / f"{img_id}_neg{i}.jpg", quality=90)
            neg_count += 1

    print(f"[{acne04_split}]  acne={pos_count}  no_acne={neg_count}  skipped(tiny)={skipped}  rejected(bad)={rejected}")

print("Starting patch extraction...")
for acne04_split, patch_split in SPLITS.items():
    extract_patches(acne04_split, patch_split)
print("Done.")

In [ ]:
for split in ["train", "val"]:
    for cls in ["acne", "no_acne"]:
        files = list((PATCH_DIR / split / cls).glob("*.jpg"))
        print(f"  data/patches/{split}/{cls}: {len(files)} images")

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for row, cls in enumerate(["acne", "no_acne"]):
    files = random.sample(list((PATCH_DIR / "train" / cls).glob("*.jpg")), 4)
    for col, f in enumerate(files):
        axes[row][col].imshow(Image.open(f))
        axes[row][col].set_title(cls, fontsize=9)
        axes[row][col].axis("off")
plt.suptitle("Sample patches — train set", fontsize=13)
plt.tight_layout()
plt.show()

---
## Section 3 — Train EfficientNet-B0 Classifier

Fine-tunes EfficientNet-B0 (pretrained on ImageNet) on the ACNE04 patches.
Training augmentation (color jitter, random crop, blur) helps the model generalise
to DermNet's different lighting and color profile.

**Output:** `outputs/classifier/best.pth`

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
from pathlib import Path

PATCH_DIR   = Path("data/patches")
CLF_OUT     = Path("outputs/classifier")
CLF_OUT.mkdir(parents=True, exist_ok=True)

EPOCHS      = 20
BATCH       = 64
LR          = 1e-4
NUM_WORKERS = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(PATCH_DIR / "train", transform=train_tf)
val_ds   = datasets.ImageFolder(PATCH_DIR / "val",   transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

class_names = train_ds.classes
print(f"Classes : {class_names}")
print(f"Train   : {len(train_ds)} images")
print(f"Val     : {len(val_ds)} images")

In [ ]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(model.classifier[1].in_features, 2),
)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
print("EfficientNet-B0 ready.")

In [ ]:
train_losses, val_losses, val_accs = [], [], []
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    avg_train = total_loss / len(train_loader)
    train_losses.append(avg_train)

    model.eval()
    total_loss = correct = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            total_loss += criterion(out, labels).item()
            correct += (out.argmax(1) == labels).sum().item()
    avg_val = total_loss / len(val_loader)
    val_acc = correct / len(val_ds)
    val_losses.append(avg_val)
    val_accs.append(val_acc)

    print(f"Epoch [{epoch:02d}/{EPOCHS}]  train={avg_train:.4f}  val={avg_val:.4f}  acc={val_acc:.4f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CLF_OUT / "best.pth")
        print(f"  checkpoint saved (acc={val_acc:.4f})")

torch.save(model.state_dict(), CLF_OUT / "last.pth")
print(f"\nDone. Best val accuracy: {best_val_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, label="Train loss")
axes[0].plot(val_losses,   label="Val loss")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(val_accs, label="Val accuracy", color="green")
axes[1].axhline(best_val_acc, linestyle="--", color="grey",
                label=f"Best: {best_val_acc:.4f}")
axes[1].set_title("Validation Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.suptitle("EfficientNet-B0 — Training on ACNE04 Patches", fontsize=13)
plt.tight_layout()
OUT_FIG = Path("outputs/figures")
OUT_FIG.mkdir(parents=True, exist_ok=True)
plt.savefig(OUT_FIG / "classifier_training_curves.png", dpi=150)
plt.show()
print("Saved -> outputs/figures/classifier_training_curves.png")

In [ ]:
# Download DermNet from Kaggle
import os
import shutil
from pathlib import Path

os.environ["KAGGLE_USERNAME"] = "your_username"   # <- paste yours
os.environ["KAGGLE_KEY"]      = "your_key"        # <- paste yours

!pip install -q kagglehub
import kagglehub

path = kagglehub.dataset_download("shubhamgoel27/dermnet")
print("Downloaded to:", path)

DERMNET_DIR = Path("data/dermnet")
DERMNET_DIR.mkdir(parents=True, exist_ok=True)
shutil.copytree(path, str(DERMNET_DIR), dirs_exist_ok=True)
print(f"DermNet ready at {DERMNET_DIR}")

---
## Section 4 — DermNet Evaluation

Evaluates the trained classifier on the DermNet test set.

**Class mapping:** `Acne and Rosacea Photos` → acne (1), all other 22 conditions → non-acne (0)

**Note on class imbalance:** DermNet test set is 312 acne vs 3,690 non-acne (8% vs 92%).
A naive model predicting non-acne always would score 92% accuracy.
F1 (acne class) and AUROC are the meaningful metrics here.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
from pathlib import Path
from PIL import Image

DERMNET_DIR = Path("data/dermnet")
CLF_OUT     = Path("outputs/classifier")
ACNE_FOLDER = "Acne and Rosacea Photos"
CONF        = 0.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
class DermNetBinary(Dataset):
    """DermNet test set with binary acne / non-acne labels."""
    def __init__(self, split, transform):
        self.samples = []
        root = DERMNET_DIR / split
        for folder in sorted(root.iterdir()):
            label = 1 if folder.name == ACNE_FOLDER else 0
            for img_path in sorted(folder.glob("*")):
                if img_path.suffix.lower() in (".jpg", ".jpeg", ".png"):
                    self.samples.append((img_path, label))
        self.transform = transform

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label

test_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_ds     = DermNetBinary("test", test_tf)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False,
                         num_workers=4, pin_memory=True)
print(f"Test set: {len(test_ds)} images")
acne_count = sum(1 for _, l in test_ds.samples if l == 1)
print(f"  acne={acne_count}  non_acne={len(test_ds)-acne_count}")

In [ ]:
# Load best checkpoint
model = models.efficientnet_b0(weights=None)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(model.classifier[1].in_features, 2),
)
model.load_state_dict(torch.load(str(CLF_OUT / "best.pth"),
                                  map_location=device, weights_only=False))
model.to(device).eval()
print("Model loaded.")

In [ ]:
# Run inference
all_probs, all_preds, all_labels = [], [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        probs = torch.softmax(model(imgs), dim=1)[:, 1]  # prob of acne
        preds = (probs >= CONF).long()
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_probs  = np.array(all_probs)
all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc   = accuracy_score(all_labels, all_preds)
f1    = f1_score(all_labels, all_preds, pos_label=1, zero_division=0)
auroc = roc_auc_score(all_labels, all_probs)

print(f"Accuracy : {acc:.4f}  (naive baseline: {1 - acne_count/len(test_ds):.4f})")
print(f"F1 (acne): {f1:.4f}")
print(f"AUROC    : {auroc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
ConfusionMatrixDisplay(cm, display_labels=["non-acne", "acne"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Confusion Matrix — DermNet Test Set")

# ROC curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
axes[1].plot(fpr, tpr, label=f"AUROC = {auroc:.4f}")
axes[1].plot([0,1],[0,1], "k--", label="Random")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve — DermNet Test Set")
axes[1].legend()

plt.tight_layout()
OUT_FIG = Path("outputs/figures")
OUT_FIG.mkdir(parents=True, exist_ok=True)
plt.savefig(OUT_FIG / "dermnet_evaluation.png", dpi=150)
plt.show()

import json
with open("outputs/dermnet_results.json", "w") as f:
    json.dump({"accuracy": acc, "f1_acne": f1, "auroc": auroc}, f, indent=2)
print("Saved -> outputs/dermnet_results.json")

---
## Section 5 — Domain Adaptation

ACNE04 (face selfies) and DermNet (clinical photos) differ in lighting,
color profile, and scale. We bridge this gap through augmentation baked
into training:

| Technique | Purpose |
|---|---|
| `ColorJitter(brightness, contrast, saturation, hue)` | Robust to lighting differences |
| `RandomResizedCrop(scale=0.7-1.0)` | Robust to scale/zoom variation |
| `GaussianBlur` | Robust to sharpness differences |
| `RandomHorizontalFlip` | Left/right invariance |

This section visualises the domain gap and the effect of augmentation.

In [ ]:
import random
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

PATCH_DIR   = Path("data/patches")
ACNE_FOLDER = "Acne and Rosacea Photos"
random.seed(42)

acne04_samples  = random.sample(list((PATCH_DIR / "train" / "acne").glob("*.jpg")), 4)
dermnet_samples = random.sample(list((DERMNET_DIR / "train" / ACNE_FOLDER).glob("*")), 4)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for col, f in enumerate(acne04_samples):
    axes[0][col].imshow(Image.open(f))
    axes[0][col].set_title("ACNE04 patch", fontsize=9)
    axes[0][col].axis("off")
for col, f in enumerate(dermnet_samples):
    axes[1][col].imshow(Image.open(f))
    axes[1][col].set_title("DermNet acne", fontsize=9)
    axes[1][col].axis("off")

plt.suptitle("Domain Gap — ACNE04 (top) vs DermNet (bottom)", fontsize=13)
plt.tight_layout()
OUT_FIG = Path("outputs/figures")
OUT_FIG.mkdir(parents=True, exist_ok=True)
plt.savefig(OUT_FIG / "domain_gap.png", dpi=150)
plt.show()
print("Saved -> outputs/figures/domain_gap.png")

In [ ]:
# Show augmentation applied to same patch 5 ways
aug_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
])

src_img = Image.open(acne04_samples[0]).convert("RGB")

fig, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(src_img)
axes[0].set_title("Original", fontsize=9); axes[0].axis("off")
for i in range(1, 6):
    axes[i].imshow(aug_tf(src_img))
    axes[i].set_title(f"Augmented {i}", fontsize=9); axes[i].axis("off")

plt.suptitle("Training Augmentation — same patch, 5 random transforms", fontsize=12)
plt.tight_layout()
plt.savefig(OUT_FIG / "augmentation_examples.png", dpi=150)
plt.show()
print("Saved -> outputs/figures/augmentation_examples.png")

In [ ]:
# RGB channel statistics — quantify domain gap
import numpy as np

def channel_stats(paths, n=200):
    sample = random.sample(list(paths), min(n, len(list(paths))))
    means = [np.array(Image.open(p).convert("RGB").resize((224,224)),
                      dtype=float).mean(axis=(0,1)) / 255.0
             for p in sample]
    means = np.array(means)
    return means.mean(axis=0), means.std(axis=0)

a_mean, a_std = channel_stats(list((PATCH_DIR / "train" / "acne").glob("*.jpg")))
d_mean, d_std = channel_stats(list((DERMNET_DIR / "train" / ACNE_FOLDER).glob("*")))

x = np.arange(3)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - 0.2, a_mean, 0.35, yerr=a_std, label="ACNE04", color="#FF6B6B", capsize=4)
ax.bar(x + 0.2, d_mean, 0.35, yerr=d_std, label="DermNet", color="#4ECDC4", capsize=4)
ax.set_xticks(x); ax.set_xticklabels(["R", "G", "B"])
ax.set_ylabel("Mean pixel value (0-1)")
ax.set_title("RGB Channel Statistics — ACNE04 vs DermNet")
ax.legend(); plt.tight_layout()
plt.savefig(OUT_FIG / "channel_stats.png", dpi=150)
plt.show()
print(f"ACNE04  mean RGB: {a_mean.round(3)}")
print(f"DermNet mean RGB: {d_mean.round(3)}")
print(f"Difference      : {(d_mean - a_mean).round(3)}")